# Wikidata QID → Text → Vector → Similar Items

This notebook takes a Wikidata item QID, converts it to text with the public `wd-textify` API, embeds that text with Jina embeddings, and queries the configured Astra vector database for similar item chunks.

Expected credentials:
- Jina: `API_tokens/jina_api.json` or `JINA_API_KEY`
- Astra DB: `API_tokens/datastax_api.json` or `ASTRA_DB_APPLICATION_TOKEN`, `ASTRA_DB_API_ENDPOINT`, and `ASTRA_COLLECTION_PREFIX`


In [10]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from textwrap import shorten

import requests

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.JinaAI import JinaAIAPIEmbedder
from src.wikidataVectorDB import AstraDBConnect


## Configure the query

Change `QID` to any Wikidata item ID, for example `Q42`, `Q90`, or `Q937`.

In [47]:
QID = "Q140263936"
LANG = "en"
TOP_K = 50

JINA_API_PATH = PROJECT_ROOT / "API_tokens" / "jina_api.json"
ASTRA_API_PATH = PROJECT_ROOT / "API_tokens" / "datastax_api.json"

WDTEXTIFY_API = "https://wd-textify.wmcloud.org/"


## Textify a Wikidata item

This calls the public `wd-textify` API, equivalent to `https://wd-textify.wmcloud.org/?id=Q42&lang=en&format=text`, and uses the returned text directly for embedding.


In [52]:
def wd_textify(qid: str, lang: str = LANG, timeout: int = 30) -> str:
    response = requests.get(
        WDTEXTIFY_API,
        params={
            "id": qid,
            "lang": lang,
            "format": "text",
            "external_ids": False,
        },
        timeout=timeout,
    )
    response.raise_for_status()

    text = response.json()[qid]
    if not text:
        raise ValueError(f"wd-textify returned empty text for {qid!r} in language {lang!r}.")
    return text


item_text = wd_textify(QID, LANG)
print(item_text[:4_000])
item_text = """Vuoden projekti, Finnish construction competition. Attributes include:
- instance of: prize
- country: Finland"""

Vuoden projekti. Attributes include:
- instance of: prize
- country: Finland


## Embed the text

Use the query embedding task because the generated vector is used to search existing document vectors.

In [56]:
embedder = JinaAIAPIEmbedder(config_path=str(JINA_API_PATH))
query_vector = embedder.embed_query(item_text)
# query_vector = embedder.embed_documents([item_text])[0]

len(query_vector), query_vector[:5]

(512,
 [-0.019210554659366608,
  -0.13163746893405914,
  0.10613199323415756,
  0.08466141670942307,
  -0.008877710439264774])

## Query Astra vector database

This searches the item collection for nearest vectors. It excludes the same QID from the printed results when possible, because the database may already contain chunks for the input entity.

In [57]:
def query_similar_items(vector: list[float], qid: str = QID, top_k: int = TOP_K, lang: str = LANG) -> list[dict]:
    astra = AstraDBConnect(lang=lang, entity_type="items", config_path=str(ASTRA_API_PATH))
    cursor = astra.item_collection.find(
        {},
        sort={"$vector": vector},
        limit=top_k + 5,
        include_similarity=True,
        projection={"$vector": False},
    )

    results = []
    for document in cursor:
        metadata = document.get("metadata", {})
        if metadata.get("QID") == qid:
            continue
        results.append(document)
        if len(results) >= top_k:
            break
    return results


similar_items = query_similar_items(query_vector, lang='en')
len(similar_items)


50

In [58]:
for rank, document in enumerate(similar_items, start=1):
    metadata = document.get("metadata", {})
    similarity = document.get("$similarity")
    qid = metadata.get("QID", "")
    label = metadata.get("Label", "")
    description = metadata.get("Description", "")
    chunk_id = metadata.get("ChunkID", "")
    preview = shorten(document.get("content", "").replace("\n", " "), width=220, placeholder=" …")

    score = f" similarity={similarity:.4f}" if isinstance(similarity, float) else ""
    print(f"{rank}. {label} ({qid}) chunk={chunk_id}{score}")
    if description:
        print(f"   {description}")
    print(f"   {preview}\n")


1. Finlandia award for architecture (Q18693941) chunk=1 similarity=0.8156
   Finnish architecture award
   Finlandia award for architecture, Finnish architecture award. Attributes include: - instance of: award. - inception: 2011. - country: Finland. - conferred by: Finnish Association of Architects. - prize money: +0 Euro.

2. Wooden Prize of the Year (Q18486953) chunk=1 similarity=0.8017
   architecture prize in Finland
   Wooden Prize of the Year, architecture prize in Finland, also known as Vuoden puupalkinto. Attributes include: - instance of: architecture award. - inception: 1994. - country: Finland. - conferred by: Puuinfo. - has …

3. Finnish State Prize for Visual Arts (Q18694006) chunk=1 similarity=0.8016
   prize for visual artists awarded annually by the State of Finland
   Finnish State Prize for Visual Arts, prize for visual artists awarded annually by the State of Finland. Attributes include: - instance of: class of award. - subclass of: Finnish State Prize. - inception: 